In [2]:
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split

project_root = Path.cwd().parent
sys.path.append(str(project_root / 'src'))

from ml_from_scratch.trees.random_forest import RandomForestClassifier
from ml_from_scratch.preprocessing.scaler import MyStandardScaler

from ml_from_scratch.metrics.classification import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix,
    log_loss
)

In [3]:
data = load_breast_cancer()

X = data.data
y = data.target

print('X shape:', X.shape)
print('y shape:', y.shape)
print('Classes:', np.unique(y))

X shape: (569, 30)
y shape: (569,)
Classes: [0 1]


In [4]:
X_train, X_test, y_train, y_test = train_test_split(X, y,
                    test_size = 0.2, random_state = 22, stratify = y)

In [5]:
scaler = MyStandardScaler()

X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

In [7]:
MyRF_model = RandomForestClassifier(n_estimators = 100, max_depth = 10,
    min_samples_split = 2, max_features = 'sqrt', random_state = 22)

In [9]:
MyRF_model.fit(X_train_scaled, y_train)

In [10]:
y_pred = MyRF_model.predict(X_test_scaled).reshape(-1)

In [12]:
accuracy = accuracy_score(y_test, y_pred)
precision = precision_score(y_test, y_pred)
recall = recall_score(y_test, y_pred)
f1 = f1_score(y_test, y_pred)

cm = confusion_matrix(y_test, y_pred)

print("Confusion Matrix:")
print(cm)

print(f"Accuracy:  {accuracy:.4f}")
print(f"Precision: {precision:.4f}")
print(f"Recall:    {recall:.4f}")
print(f"F1 Score:  {f1:.4f}")

Confusion Matrix:
[[38  4]
 [ 4 68]]
Accuracy:  0.9298
Precision: 0.9444
Recall:    0.9444
F1 Score:  0.9444


In [13]:
from sklearn.ensemble import RandomForestClassifier as SklearnRandomForest

sklearn_model = SklearnRandomForest(
    n_estimators = 100,
    max_depth = 10,
    min_samples_split = 2,
    max_features = 'sqrt',
    random_state = 42
)

In [15]:
sklearn_model.fit(X_train_scaled, y_train)

sklearn_pred = sklearn_model.predict(X_test_scaled)

In [16]:
custom_metrics = {
    
    'Accuracy': accuracy_score(y_test, y_pred),
    'Precision': precision_score(y_test, y_pred),
    'Recall': recall_score(y_test, y_pred),
    'F1': f1_score(y_test, y_pred)
}

sklearn_metrics = {
    'Accuracy': accuracy_score(y_test, sklearn_pred),
    'Precision': precision_score(y_test, sklearn_pred),
    'Recall': recall_score(y_test, sklearn_pred),
    'F1': f1_score(y_test, sklearn_pred)
}

comparison = pd.DataFrame(
    {
        'My Random Forest': custom_metrics,
        'Sklearn Random Forest': sklearn_metrics,
    }
)

comparison

,My Random Forest,Sklearn Random Forest
Accuracy,0.929825,0.947368
Precision,0.944444,0.945946
Recall,0.944444,0.972222
F1,0.944444,0.958904


# Final Interpretation

The custom Random Forest achieved strong classification performance on the Breast Cancer Wisconsin dataset.

On the test set, the model achieved 92.98% accuracy, 94.44% precision, 94.44% recall, and a 94.44% F1 score. The confusion matrix was:

[[38  4]
 [ 4 68]]

This means that the model correctly classified 38 samples from class 0 and 68 samples from class 1, while producing 4 false positives and 4 false negatives. In this medical classification problem, false negatives are particularly important because they represent positive cases incorrectly classified as negative.

The custom implementation was compared with sklearn's RandomForestClassifier using the same general configuration. The sklearn model achieved 94.74% accuracy, 94.59% precision, 97.22% recall, and 95.89% F1 score.

The two implementations produced similar overall performance, although the sklearn model achieved higher accuracy, recall, and F1 on this particular test split. Differences are expected because the custom implementation and sklearn may differ in implementation details, optimization strategies, and internal handling of tree construction and randomness.

The comparison should therefore be considered a reference for validating the custom implementation rather than evidence that one implementation is universally better than the other.

Overall, this experiment demonstrated the implementation of a Random Forest classifier from scratch using NumPy, including bootstrap sampling, random feature selection, training multiple Decision Trees, majority voting, probability estimation, evaluation, and comparison with a standard machine learning library.